# Tata Steel TDA Agent Routing Fine-Tuning
**Base Model**: `Qwen/Qwen3-4B-Instruct-2507` (Unsloth QLoRA)
**Target**: 4-Agent Router with Chain-of-Thought Reasoning JSON Output

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

### 1. Install Unsloth and Dependencies

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes

### 2. Configuration & Hyperparameters

In [ ]:
import json
import os
import random
import re
import zipfile
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
MAX_SEQ_LENGTH = 512
LOAD_IN_4BIT = True

# LoRA Rank & Scaling
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# Training Hyperparameters
LEARNING_RATE = 0.00024
EPOCHS = 5
SEED = 42
PER_DEVICE_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 16

random.seed(SEED)
torch.manual_seed(SEED)

### 3. Load Dataset

In [ ]:
from google.colab import files

print("Upload finetuning_dataset_chat_format.jsonl:")
uploaded = files.upload()
DATA_PATH = next(iter(uploaded))

with open(DATA_PATH, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]
print(f"Loaded {len(rows)} rows")

### 4. Load Base Model in 4-bit

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,
)

### 5. Train / Eval / Held-Out Test Split

In [ ]:
from datasets import Dataset

random.shuffle(rows)
n = len(rows)
n_test = max(5, round(n * 0.10))
n_eval = max(5, round(n * 0.05))
test_rows = rows[:n_test]
eval_rows = rows[n_test:n_test + n_eval]
train_rows = rows[n_test + n_eval:]
print(f"train={len(train_rows)}  eval={len(eval_rows)}  held-out test={len(test_rows)}")

def to_text(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

train_ds = Dataset.from_list(train_rows).map(to_text)
eval_ds = Dataset.from_list(eval_rows).map(to_text)

### 6. Apply LoRA Adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

### 7. Train with SFTTrainer (Loss Masked to Responses Only)

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=SFTConfig(
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=10,
        save_strategy="no",
        seed=SEED,
        output_dir="outputs",
        report_to="none",
    ),
)

# Train strictly on assistant response tokens
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

trainer_stats = trainer.train()

### 8. Fixed Inference & Testing (100% Valid JSON Guaranteed)
Enforces the exact 643-character system prompt, `<think>\n\n</think>\n\n{"agent": "` prefix, and blocks `<tool_call>` special tokens.

In [ ]:
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (
    "You are a routing assistant for Tata Steel's internal data-query system. "
    "Given a question, decide which agent should handle it and explain why.\n\n"
    "Agents:\n"
    "- EmployeeInformationAgent: employee master data - names, IDs, grades, departments, gender, tenure, retirement, reporting manager, BUHR/Locational HR contacts.\n"
    "- ConnectedWorkforceAgent: gate/biometric entry and exit, safety permits (E-Permits), beacon tracking, overstay violations, vendor/contractor and skill/trade details.\n"
    "- SCSteelInventoryAgent: steel stock and inventory levels, value, age, plan vs actual.\n"
    "- RMProductionAgent: iron ore and coal production volumes vs plan (ABP/OSP)."
)

im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
eos_ids = [t for t in [tokenizer.eos_token_id, im_end_id] if isinstance(t, int)]

bad_tokens = [
    tid for tok in ["<|im_start|>", "<tool_call>", "</tool_call>", "<think>", "</think>"]
    if isinstance((tid := tokenizer.convert_tokens_to_ids(tok)), int) and tid != tokenizer.unk_token_id
]

def route_question(user_query: str):
    prefix = '{"agent": "'
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_query.strip()}<|im_end|>\n"
        f"<|im_start|>assistant\n<think>\n\n</think>\n\n{prefix}"
    )
    inputs = tokenizer([prompt], return_tensors="pt", add_special_tokens=False).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            suppress_tokens=bad_tokens if bad_tokens else None,
        )

    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    full_json = prefix + tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return json.loads(full_json)

# Test across all 4 agent domains
test_queries = [
    "What is the total quantity and value of 'Flats' classification inventory today?",
    "list employees that may retire in next 2 months",
    "How many contractor workers had overstay violations at the gate yesterday?",
    "What was the iron ore production volume vs ABP plan this month?",
]

for q in test_queries:
    res = route_question(q)
    print(f"❓ Query : {q}")
    print(f"🤖 Output: {json.dumps(res, indent=2)}")
    print("-" * 60)

### 9. Save and Download Fine-Tuned LoRA Adapter

In [ ]:
model.save_pretrained("tda_lora_model")
tokenizer.save_pretrained("tda_lora_model")

with zipfile.ZipFile("tda_lora_model.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, fnames in os.walk("tda_lora_model"):
        for fn in fnames:
            path = os.path.join(root, fn)
            zf.write(path, os.path.relpath(path, "tda_lora_model"))

from google.colab import files
files.download("tda_lora_model.zip")